# 01 · Language basics

This chapter covers core Python with no data frames involved. Almost all of it maps onto base R, and it is the foundation that pandas sits on.

1. Assignment and types
2. Lists, dicts and tuples
3. Indexing
4. Functions
5. Control flow and iteration
6. Vectorization with NumPy

## 1. Assignment and types

**In R**

```r
x <- 5L
y <- 2.5
name <- "Ada"
is_ready <- TRUE

class(y)
```

**In Python**

In [ ]:
x = 5
y = 2.5
name = "Ada"
is_ready = True

type(y)

| R | Python | Notes |
| --- | --- | --- |
| integer (`5L`) | `int` (`5`) | No `L` suffix. A number without a decimal point is an `int`. |
| numeric (`2.5`) | `float` (`2.5`) | |
| character | `str` | Single or double quotes, as in R. |
| logical (`TRUE`, `FALSE`) | `bool` (`True`, `False`) | Capitalised only the first letter. No `T` / `F` shorthand. |
| `NULL` | `None` | |
| `NA` | no built-in equivalent | Missing values belong to pandas and NumPy. See chapter 02. |

One difference runs deeper than spelling. In R, `5` is a vector of length one. In Python, `5` is just a number: it has no length and you cannot index into it.

### Operators

Most operators are the same (`+`, `-`, `*`, `/`, `==`, `!=`, `<`, `>=`). These are the ones that differ:

| | R | Python |
| --- | --- | --- |
| power | `2^10` | `2 ** 10` |
| remainder | `7 %% 2` | `7 % 2` |
| integer division | `7 %/% 2` | `7 // 2` |
| and / or / not | `&&`, `\|\|`, `!` | `and`, `or`, `not` |
| membership | `x %in% y` | `x in y` |

In [ ]:
print(2 ** 10)
print(7 / 2)       # ordinary division gives 3.5, as in R
print(7 // 2)
print(7 % 2)
print(x > 3 and not is_ready)

**Watch out:** `^` is a valid Python operator, but it means something unrelated (bitwise XOR). `2 ^ 10` runs without error and gives `8`. Always use `**` for powers.

### Strings

**In R**

```r
paste0("Hello, ", name, "!")
glue::glue("Hello, {name}!")
sprintf("%.2f", 2 / 3)
nchar(name)
toupper(name)
```

**In Python**

In [ ]:
print("Hello, " + name + "!")    # + joins strings
print(f"Hello, {name}!")         # an f-string: works like glue
print(f"{2 / 3:.2f}")            # formatting goes after a colon
print(len(name))
print(name.upper())

The **f-string** (note the `f` before the opening quote) is the one to remember. Anything inside `{ }` is evaluated and inserted.

Unlike `paste0()`, `+` will not convert numbers for you: `"Total: " + 5` is an error. Use an f-string instead: `f"Total: {5}"`.

## 2. Lists, dicts and tuples

R has one workhorse container, the vector. Python splits that job up:

| R | Python | Written as |
| --- | --- | --- |
| `list(1, "a", TRUE)` | **list** | `[1, "a", True]` |
| `list(a = 1, b = 2)`, or a named vector | **dict** | `{"a": 1, "b": 2}` |
| (no equivalent) | **tuple** | `(1, 2)` |
| `c(1, 2, 3)`, an atomic vector | **NumPy array** (section 6) | `np.array([1, 2, 3])` |

### Lists

**In R**

```r
nums <- c(3, 1, 2)
nums <- c(nums, 10)
length(nums)
sort(nums)
5 %in% nums
```

**In Python**

In [ ]:
nums = [3, 1, 2]
nums.append(10)        # changes nums itself: no reassignment needed

print(nums)
print(len(nums))
print(sorted(nums))
print(5 in nums)

A Python list looks like `c()` but behaves like an R `list()`:

- It can hold mixed types without converting them. `c(1, "a", TRUE)` turns everything into character; `[1, "a", True]` keeps each item as it is.
- **It is not vectorized.** Arithmetic on a list does not work element by element:

In [ ]:
print(nums * 2)       # repeats the list; does NOT double each element
print(nums + [4])     # joins two lists; does NOT add 4 to each element

For element-wise arithmetic you need a NumPy array, covered in section 6.

Notice also that `nums.append(10)` changed `nums` without any assignment. R never modifies an object behind your back; Python often does. Chapter 07 covers this properly.

### Dicts

A dict maps keys to values, like a named list.

**In R**

```r
mass <- list(Adelie = 3701, Gentoo = 5076)
mass$Adelie
mass[["Chinstrap"]] <- 3733
names(mass)
"Gentoo" %in% names(mass)
```

**In Python**

In [ ]:
mass = {"Adelie": 3701, "Gentoo": 5076}

print(mass["Adelie"])
mass["Chinstrap"] = 3733
print(list(mass))            # the keys, like names()
print("Gentoo" in mass)      # `in` checks the keys
print(mass)

Values are looked up by key only. There is no `mass[[1]]`: a dict has no positions.

Dicts appear constantly as function arguments in pandas, for example `df.rename(columns={"old": "new"})`.

### Tuples

A tuple is a list that cannot be changed after it is created. You will rarely build one on purpose, but functions often return them. In chapter 00, `penguins.shape` gave back the tuple `(344, 8)`.

The useful trick is **unpacking** a tuple into separate variables:

In [ ]:
shape = (344, 8)
n_rows, n_cols = shape

print(n_rows)
print(n_cols)

## 3. Indexing

This is the part that needs retraining. Two rules:

1. Counting starts at **0**.
2. A slice `a:b` **includes `a` and excludes `b`**.

**In R**

```r
x <- c("a", "b", "c", "d", "e")

x[1]         # "a"
x[2:4]       # "b" "c" "d"
x[-1]        # everything except the first
head(x, 3)   # "a" "b" "c"
tail(x, 2)   # "d" "e"
rev(x)
```

**In Python**

In [ ]:
x = ["a", "b", "c", "d", "e"]

print(x[0])       # the first element
print(x[1:4])     # positions 1, 2 and 3
print(x[-1])      # the LAST element (not "drop the first")
print(x[1:])      # everything except the first
print(x[:3])      # the first three
print(x[-2:])     # the last two
print(x[::-1])    # reversed

It helps to picture the numbers as pointing at the gaps *between* elements:

```
  a   b   c   d   e
0   1   2   3   4   5
```

`x[1:4]` is everything between gap 1 and gap 4. This is why `x[:3]` gives exactly three elements, and why `x[:3]` and `x[3:]` split a list with nothing duplicated or dropped.

| To get | R | Python |
| --- | --- | --- |
| first element | `x[1]` | `x[0]` |
| last element | `x[length(x)]` | `x[-1]` |
| first three | `x[1:3]` | `x[:3]` |
| last two | `tail(x, 2)` | `x[-2:]` |
| all but the first | `x[-1]` | `x[1:]` |
| every second element | `x[c(TRUE, FALSE)]` | `x[::2]` |

Negative indices are the classic trap: `x[-1]` *drops* the first element in R and *selects the last* in Python.

Strings are indexed the same way, so `substr("penguin", 1, 3)` becomes:

In [ ]:
"penguin"[0:3]

### Sequences: `1:5` becomes `range()`

`range()` follows the same rules: it starts at 0 and stops *before* the end value. It produces its numbers lazily, so wrap it in `list()` to look at them.

In [ ]:
print(list(range(5)))           # 0 to 4: five numbers, like seq_len(5) shifted down by one
print(list(range(1, 6)))        # 1:5
print(list(range(0, 10, 2)))    # seq(0, 8, by = 2)

One more difference: indexing past the end is an error in Python (`x[10]` raises `IndexError`), where R would quietly return `NA`.

## 4. Functions

**In R**

```r
add_tax <- function(price, rate = 0.08) {
  price * (1 + rate)
}

add_tax(100)
add_tax(100, rate = 0.2)
```

**In Python**

In [ ]:
def add_tax(price, rate=0.08):
    return price * (1 + rate)


print(add_tax(100))
print(add_tax(100, rate=0.2))

What changed:

- `def name(args):` replaces `name <- function(args)`.
- **Indentation replaces `{ }`.** The body is whatever is indented under the `def` line. The convention is four spaces, and the editor inserts them for you after the colon.
- **`return` is required.** R returns the value of the last expression. A Python function without `return` gives back `None`.

That last one is the mistake nearly every R user makes in their first week:

In [ ]:
def add_tax_broken(price, rate=0.08):
    price * (1 + rate)       # calculated, then thrown away


print(add_tax_broken(100))

### Returning several values

In R you would return a list. In Python, return several values separated by commas (this makes a tuple) and unpack them on the other side.

In [ ]:
def min_max(values):
    return min(values), max(values)


low, high = min_max([3, 1, 2, 10])
print(low, high)

### Anonymous functions

`function(x) x + 1` or `\(x) x + 1` becomes `lambda x: x + 1`. A lambda is limited to a single expression.

In [ ]:
add_one = lambda x: x + 1
add_one(5)

### Functions vs methods, again

Chapter 00 introduced methods: functions attached to an object and called with a dot. Some methods return a new object, and some change the object in place and return `None`. Lists have one of each for sorting:

In [ ]:
nums = [3, 1, 2]

print(sorted(nums))    # a function: returns a new sorted list
print(nums)            # the original is untouched

result = nums.sort()   # a method: sorts nums in place...
print(result)          # ...and returns None
print(nums)

So `nums = nums.sort()`, the natural thing to write coming from R, replaces your list with `None`. If a variable is unexpectedly `None`, this is the first thing to check.

## 5. Control flow and iteration

### if / else

**In R**

```r
temp <- 18

if (temp > 25) {
  label <- "hot"
} else if (temp > 10) {
  label <- "mild"
} else {
  label <- "cold"
}
```

**In Python**

In [ ]:
temp = 18

if temp > 25:
    label = "hot"
elif temp > 10:
    label = "mild"
else:
    label = "cold"

print(label)

No parentheses around the condition, a colon at the end of each line that opens a block, and `elif` in place of `else if`.

For a one-line choice between two values, R's `if (cond) a else b` becomes `a if cond else b`:

In [ ]:
size = "big" if temp > 20 else "small"
size

### for loops

**In R**

```r
species <- c("Adelie", "Gentoo", "Chinstrap")

for (s in species) {
  print(s)
}

for (i in seq_along(species)) {
  cat(i, species[i], "\n")
}
```

**In Python**

In [ ]:
species = ["Adelie", "Gentoo", "Chinstrap"]

for s in species:
    print(s)

for i, s in enumerate(species):    # enumerate gives (position, item) pairs
    print(i, s)

Looping over a dict with `.items()` gives each key and value together:

In [ ]:
for name, grams in mass.items():
    print(f"{name}: {grams} g")

### List comprehensions: Python's `sapply`

**In R**

```r
nums <- c(3, 1, 2, 10)

sapply(nums, function(x) x^2)       # or purrr::map_dbl(nums, \(x) x^2)
Filter(function(x) x > 2, nums)     # or purrr::keep(nums, \(x) x > 2)
```

**In Python**

In [ ]:
nums = [3, 1, 2, 10]

squares = [x ** 2 for x in nums]
big = [x for x in nums if x > 2]

print(squares)
print(big)

Read `[x ** 2 for x in nums]` as "x squared, for each x in nums". An optional `if` at the end filters. The same idea builds a dict with curly braces:

In [ ]:
mass_kg = {name: grams / 1000 for name, grams in mass.items()}
mass_kg

R teaches you to avoid loops. In Python, loops and comprehensions over lists are ordinary, everyday code. For arithmetic on whole columns of numbers, though, you still want vectorization, which is the next section.

## 6. Vectorization with NumPy

A NumPy array is the real counterpart to an R atomic vector: every element has the same type, and arithmetic works element by element.

**In R**

```r
v <- c(1, 2, 3, 4)

v * 2
v + v
v[v > 2]
mean(v)
ifelse(v > 2, "big", "small")
seq(0, 1, length.out = 5)
```

**In Python**

In [ ]:
import numpy as np

v = np.array([1, 2, 3, 4])

print(v * 2)
print(v + v)
print(v[v > 2])                           # logical subsetting, as in R
print(v.mean())
print(np.where(v > 2, "big", "small"))    # ifelse()
print(np.linspace(0, 1, 5))               # seq(0, 1, length.out = 5)

This should feel like home. Indexing still starts at 0, but logical subsetting, element-wise arithmetic and summary functions work as they do in R.

Like an atomic vector, an array has a single type:

In [ ]:
print(v.dtype)
print(np.array([1, 2.5, 3]).dtype)    # ints are promoted to float, as in c(1L, 2.5)

### Combining conditions

R uses `&` and `|` for element-wise logic and `&&` and `||` for single values. Python has the same split with different spelling: `&`, `|` and `~` for arrays, and `and`, `or` and `not` for single values.

Each condition must be wrapped in parentheses:

In [ ]:
print(v[(v > 1) & (v < 4)])
print(v[~(v > 2)])              # ~ is element-wise "not", like ! in R

Using `and` here (`v > 1 and v < 4`) raises an error about the truth value of an array being ambiguous. When you see that message, switch to `&` and add parentheses.

### No silent recycling

In R, `c(1, 2, 3, 4) + c(10, 20)` recycles the shorter vector and returns `11 22 13 24`. NumPy refuses and raises an error, because the lengths do not match. A single number is still applied to every element, which is why `v * 2` works.

### Why this matters

A pandas data frame column is a NumPy array with labels attached. Everything in this section, from `v * 2` to `v[v > 2]` to `&` with parentheses, works the same way on columns, and that is where chapter 02 picks up.

## Recap

| R | Python |
| --- | --- |
| `x <- 5` | `x = 5` |
| `TRUE`, `FALSE`, `NULL` | `True`, `False`, `None` |
| `2^10` | `2 ** 10` |
| `paste0("a", x)` / `glue("a{x}")` | `f"a{x}"` |
| `c(1, 2, 3)` | `[1, 2, 3]` (list) or `np.array([1, 2, 3])` (vectorized) |
| `list(a = 1)` | `{"a": 1}` |
| `x[1]`, `x[2:4]` | `x[0]`, `x[1:4]` |
| `x[-1]` (drop first) | `x[1:]` |
| `tail(x, 1)` | `x[-1]` |
| `1:5` | `range(1, 6)` |
| `function(x) { x + 1 }` | `def f(x): return x + 1` |
| `if / else if / else` | `if / elif / else` |
| `sapply(x, f)` | `[f(i) for i in x]` |
| `ifelse(cond, a, b)` | `np.where(cond, a, b)` |
| `&`, `\|`, `!` on vectors | `&`, `\|`, `~` on arrays, with parentheses |

---
Next: work through `exercises/01_language_basics.ipynb`, then move on to `02_dataframes.ipynb`.